# Site Details

This notebook performs comprehensive data quality checks on the Site_Details dataset.


## 1. Data Loading


In [1]:
import pandas as pd

# Load the Site_Details data
df = pd.read_csv('../datasets/Site_Details.csv')

print("Data loaded successfully!")
print(f"Total records: {len(df)}")


Data loaded successfully!
Total records: 50


## 2. Basic Data Exploration


In [2]:
# Display first few rows
df.head()


,Site ID,Site Name,Site Format,Region,City,State,Store Size,Open Date,Status
0,JI0Y,Smart Mumbai,Digital,North,Mumbai,Maharashtra,6041,03-05-2015,Active
1,VUZZ,Digital Ahmedabad,Fresh,North,Ahmedabad,Gujarat,19719,10-08-2021,Inactive
2,PKH1,Smart Mumbai,Smart,North,Mumbai,Maharashtra,17449,01-02-2016,Inactive
3,30T9,Digital Hubli,Smart,South,Hubli,Karnataka,23089,15-04-2018,Inactive
4,CX19,Fresh New Delhi,Smart,North,New Delhi,Delhi,17455,12-02-2018,Inactive


In [3]:
# Display column names
df.columns


Index(['Site ID', 'Site Name', 'Site Format', 'Region', 'City', 'State',
       'Store Size', 'Open Date', 'Status'],
      dtype='object')

In [4]:
# Display dataset shape
df.shape


(50, 9)

In [5]:
# Display data types and basic info
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Site ID      50 non-null     object
 1   Site Name    50 non-null     object
 2   Site Format  50 non-null     object
 3   Region       50 non-null     object
 4   City         50 non-null     object
 5   State        50 non-null     object
 6   Store Size   50 non-null     int64 
 7   Open Date    50 non-null     object
 8   Status       50 non-null     object
dtypes: int64(1), object(8)
memory usage: 3.6+ KB


## 3. Data Quality Checks

### 3.1 Missing Values Analysis


In [6]:
# Check for missing values
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage': missing_percentage
})

print("=== Missing Values Analysis ===")
print(missing_df)
print(f"\nTotal missing values: {df.isnull().sum().sum()}")


=== Missing Values Analysis ===
             Missing Values  Percentage
Site ID                   0         0.0
Site Name                 0         0.0
Site Format               0         0.0
Region                    0         0.0
City                      0         0.0
State                     0         0.0
Store Size                0         0.0
Open Date                 0         0.0
Status                    0         0.0

Total missing values: 0


### 3.2 Duplicate Records Check


In [7]:
# Check for duplicate records
duplicates_all = df.duplicated().sum()
print("=== Duplicate Records Check ===")
print(f"Total duplicate rows: {duplicates_all}")


=== Duplicate Records Check ===
Total duplicate rows: 0


### 3.3 Data Type Validation


In [8]:
# Check data types
print("=== Data Type Validation ===")
print(df.dtypes)

# Verify expected data types
expected_types = {
    'Site ID': 'object',
    'Site Name': 'object',
    'Site Format': 'object',
    'Region': 'object',
    'City': 'object',
    'State': 'object',
    'Store Size': 'int64',
    'Open Date': 'object',
    'Status': 'object'
}

type_issues = []
for col, expected in expected_types.items():
    if col in df.columns and df[col].dtype != expected:
        type_issues.append(f"{col}: expected {expected}, got {df[col].dtype}")

if type_issues:
    print("\nData Type Issues:")
    for issue in type_issues:
        print(f"  - {issue}")
else:
    print("\nAll data types are correct!")


=== Data Type Validation ===
Site ID        object
Site Name      object
Site Format    object
Region         object
City           object
State          object
Store Size      int64
Open Date      object
Status         object
dtype: object

All data types are correct!


### 3.4 Value Range Checks


In [9]:
# Check value ranges for numeric columns
print("=== Value Range Checks ===")

# Get numeric columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in numeric_cols:
    min_val = df[col].min()
    max_val = df[col].max()
    print(f"\n{col}: Min={min_val}, Max={max_val}")
    
    # Check for negative values (except where appropriate)
    if 'ID' not in col and 'Flag' not in col:
        neg_count = (df[col] < 0).sum()
        if neg_count > 0:
            print(f"  Negative values found: {neg_count}")


=== Value Range Checks ===

Store Size: Min=5235, Max=29692


### 3.5 Categorical Value Validation


In [10]:
# Check unique values in categorical columns
print("=== Categorical Value Validation ===")

cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    if 'ID' not in col and 'Date' not in col and 'Name' not in col:
        unique_vals = df[col].unique()
        print(f"\n{col} unique values: {unique_vals}")


=== Categorical Value Validation ===

Site Format unique values: ['Digital' 'Fresh' 'Smart' 'Trends']

Region unique values: ['North' 'South' 'West']

City unique values: ['Mumbai' 'Ahmedabad' 'Hubli' 'New Delhi' 'Bangalore' 'Surat' 'Coimbatore'
 'Chennai' 'Madurai' 'Pune' 'Nagpur' 'Vadodara']

State unique values: ['Maharashtra' 'Gujarat' 'Karnataka' 'Delhi' 'Tamil Nadu']

Status unique values: ['Active' 'Inactive']


### 3.6 Outlier Detection


In [11]:
# Detect outliers using IQR method
print("=== Outlier Detection (IQR Method) ===")

def detect_outliers_iqr(column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return len(outliers), lower_bound, upper_bound

# Check outliers for numeric columns
numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns
for col in numeric_columns:
    if 'ID' not in col:
        count, lower, upper = detect_outliers_iqr(col)
        print(f"\n{col}:")
        print(f"  Outliers: {count}")
        print(f"  Lower Bound: {lower:.2f}")
        print(f"  Upper Bound: {upper:.2f}")


=== Outlier Detection (IQR Method) ===

Store Size:
  Outliers: 0
  Lower Bound: -5734.12
  Upper Bound: 40730.88


## 4. Data Quality Summary


In [12]:
# Generate comprehensive data quality summary
print("=" * 50)
print("DATA QUALITY SUMMARY")
print("=" * 50)

print(f"\n1. Dataset Overview:")
print(f"   - Total Records: {len(df)}")
print(f"   - Total Columns: {len(df.columns)}")

print(f"\n2. Missing Values:")
print(f"   - Total Missing: {df.isnull().sum().sum()}")
for col in df.columns:
    missing = df[col].isnull().sum()
    print(f"   - {col}: {missing}")

print(f"\n3. Duplicate Records:")
print(f"   - Duplicate Rows: {df.duplicated().sum()}")

print(f"\n4. Data Quality Status:")
quality_passed = True

if df.isnull().sum().sum() > 0:
    print(f"   ❌ Missing values found")
    quality_passed = False
else:
    print(f"   ✓ No missing values")

if df.duplicated().sum() > 0:
    print(f"   ❌ Duplicate records found")
    quality_passed = False
else:
    print(f"   ✓ No duplicate records")

if quality_passed:
    print(f"\n✅ DATA QUALITY CHECK PASSED!")
else:
    print(f"\n❌ DATA QUALITY ISSUES DETECTED - REVIEW REQUIRED!")


DATA QUALITY SUMMARY

1. Dataset Overview:
   - Total Records: 50
   - Total Columns: 9

2. Missing Values:
   - Total Missing: 0
   - Site ID: 0
   - Site Name: 0
   - Site Format: 0
   - Region: 0
   - City: 0
   - State: 0
   - Store Size: 0
   - Open Date: 0
   - Status: 0

3. Duplicate Records:
   - Duplicate Rows: 0

4. Data Quality Status:
   ✓ No missing values
   ✓ No duplicate records

✅ DATA QUALITY CHECK PASSED!
